In [ ]:
import json
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from deep_translator import GoogleTranslator
from datetime import datetime
import re
import wave
import queue
import threading
import sounddevice as sd

from vosk import Model, KaldiRecognizer
from textblob import TextBlob

# =========================================================
# PDF IMPORTS
# =========================================================

from reportlab.lib.pagesizes import letter
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)
from reportlab.lib.styles import getSampleStyleSheet

# =========================================================
# VOSK MODEL
# =========================================================

model_path = r"C:\Users\User\Downloads\vosk-model-small-en-us-0.15"

model = Model(model_path)

# =========================================================
# GLOBALS
# =========================================================

listening = False
audio_frames = []
q = queue.Queue()

# Supported languages (can be expanded)
languages = {
    "English": "en",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Chinese (Simplified)": "zh-CN",
    "Japanese": "ja",
    "Korean": "ko",
    "Arabic": "ar",
    "Hindi": "hi",
    "Italian": "it"
}

# =========================================================
# MAIN APPLICATION
# =========================================================

class MedicalApp:

    def __init__(self, root):

        self.root = root

        self.root.title("Electronic Medical Record System")
        self.root.geometry("1000x700")

        # =====================================================
        # GLOBAL UI SETTINGS
        # =====================================================

        root.option_add("*Font", ("Segoe UI", 10))
        root.option_add("*Background", "#66B2B2")
        root.option_add("*Foreground", "white")

        # =====================================================
        # COLOURS
        # =====================================================

        self.PRIMARY = "#66B2B2"
        self.DARK = "#4C9999"
        self.LIGHT = "#A0D6D6"
        self.WHITE = "white"
        self.TEXT = "black"

        self.root.configure(bg=self.PRIMARY)

        # =====================================================
        # STYLE
        # =====================================================

        style = ttk.Style()

        style.theme_use("clam")

        style.configure(
            ".",
            background=self.PRIMARY,
            foreground=self.WHITE,
            font=("Segoe UI", 10)
        )

        style.configure(
            "TFrame",
            background=self.PRIMARY
        )

        style.configure(
            "TLabelframe",
            background=self.PRIMARY,
            bordercolor=self.WHITE,
            borderwidth=2,
            relief="solid"
        )

        style.configure(
            "TLabelframe.Label",
            background=self.PRIMARY,
            foreground=self.WHITE,
            font=("Segoe UI", 10, "bold")
        )

        style.configure(
            "TLabel",
            background=self.PRIMARY,
            foreground=self.WHITE,
            font=("Segoe UI", 10)
        )

        style.configure(
            "TButton",
            background=self.LIGHT,
            foreground=self.TEXT,
            font=("Segoe UI", 10, "bold"),
            padding=6
        )

        style.map(
            "TButton",
            background=[("active", self.DARK)],
            foreground=[("active", self.WHITE)]
        )

        style.configure(
            "TNotebook",
            background=self.PRIMARY
        )

        style.configure(
            "TNotebook.Tab",
            background=self.LIGHT,
            foreground=self.TEXT,
            font=("Segoe UI", 10, "bold"),
            padding=[12, 6]
        )

        style.map(
            "TNotebook.Tab",
            background=[("selected", self.DARK)],
            foreground=[("selected", self.WHITE)]
        )

        style.configure(
            "TCombobox",
            fieldbackground=self.WHITE,
            background=self.LIGHT,
            foreground=self.TEXT
        )

        # =====================================================
        # BUTTON STYLE
        # =====================================================

        self.button_style = {
            "font": ("Segoe UI", 10, "bold"),
            "bg": self.LIGHT,
            "fg": self.TEXT,
            "activebackground": self.DARK,
            "activeforeground": self.WHITE,
            "relief": "solid",
            "bd": 1,
            "highlightthickness": 2,
            "highlightbackground": self.WHITE,
            "highlightcolor": self.WHITE,
            "cursor": "hand2",
            "padx": 10,
            "pady": 5
        }

        # =====================================================
        # TOP FRAME
        # =====================================================

        top_frame = tk.Frame(
            root,
            bg=self.PRIMARY
        )

        top_frame.pack(
            fill="x",
            padx=10,
            pady=10
        )

        tk.Button(
            top_frame,
            text="Load Record",
            command=self.load_json,
            **self.button_style
        ).pack(side="left", padx=5)

        tk.Button(
            top_frame,
            text="Save Record",
            command=self.save_json,
            **self.button_style
        ).pack(side="left", padx=5)

        tk.Button(
            top_frame,
            text="Generate PDF",
            command=self.generate_pdf,
            **self.button_style
        ).pack(side="left", padx=5)

        current_time = datetime.now().strftime(
            "%d/%m/%Y %H:%M:%S"
        )

        tk.Label(
            top_frame,
            text=current_time,
            bg=self.PRIMARY,
            fg=self.WHITE,
            font=("Segoe UI", 10, "bold")
        ).pack(side="left", padx=15)

        # =====================================================
        # NOTEBOOK
        # =====================================================

        self.notebook = ttk.Notebook(root)

        self.notebook.pack(
            fill="both",
            expand=True,
            padx=10,
            pady=10
        )

        self.create_patient_tab()
        self.create_medical_tab()
        self.create_notes_tab()
        self.create_translator_tab()

        # Start queue updater
        self.update_textbox()

    # =========================================================
    # STYLED ENTRY
    # =========================================================

    def styled_entry(self, parent, width=40):

        return tk.Entry(
            parent,
            width=width,
            font=("Segoe UI", 10),
            bg="white",
            fg="black",
            relief="flat",
            bd=0,
            highlightthickness=2,
            highlightbackground="white",
            highlightcolor="white",
            insertbackground="black"
        )

    # =========================================================
    # STYLED TEXT
    # =========================================================

    def styled_text(self, parent, height=6):

        return tk.Text(
            parent,
            height=height,
            font=("Segoe UI", 10),
            bg="white",
            fg="black",
            relief="flat",
            bd=0,
            highlightthickness=2,
            highlightbackground="white",
            highlightcolor="white",
            insertbackground="black"
        )

    # =========================================================
    # STYLED LISTBOX
    # =========================================================

    def styled_listbox(self, parent, height=6):

        return tk.Listbox(
            parent,
            height=height,
            font=("Segoe UI", 10),
            bg="white",
            fg="black",
            relief="flat",
            bd=0,
            highlightthickness=2,
            highlightbackground="white",
            highlightcolor="white",
            selectbackground=self.DARK,
            selectforeground="white"
        )

    # =========================================================
    # STYLE TOPLEVEL
    # =========================================================

    def style_toplevel(self, window):

        window.configure(bg=self.PRIMARY)

    # =========================================================
    # PATIENT TAB
    # =========================================================

    def create_patient_tab(self):

        self.patient_tab = ttk.Frame(self.notebook)

        self.notebook.add(
            self.patient_tab,
            text="Patient Details"
        )

        frame = ttk.LabelFrame(
            self.patient_tab,
            text="Personal Information",
            padding=20
        )

        frame.pack(
            fill="x",
            padx=10,
            pady=10
        )

        self.fields = {}

        labels = [
            ("Patient ID", "patient_id"),
            ("First Name", "first_name"),
            ("Last Name", "last_name"),
            ("Email", "email"),
            ("Phone", "phone")
        ]

        for i, (label, key) in enumerate(labels):

            ttk.Label(
                frame,
                text=label
            ).grid(
                row=i,
                column=0,
                sticky="w",
                pady=12,
                padx=10
            )

            entry = self.styled_entry(frame)

            entry.grid(
                row=i,
                column=1,
                pady=12,
                padx=10,
                ipady=5
            )

            self.fields[key] = entry

        ttk.Label(
            frame,
            text="Gender"
        ).grid(
            row=len(labels),
            column=0,
            sticky="w",
            pady=12,
            padx=10
        )

        self.gender_var = tk.StringVar()

        gender_combo = ttk.Combobox(
            frame,
            textvariable=self.gender_var,
            values=["Male", "Female", "Other"],
            state="readonly",
            width=37
        )

        gender_combo.grid(
            row=len(labels),
            column=1,
            pady=12,
            padx=10
        )

        gender_combo.set("Male")

    # =========================================================
    # MEDICAL TAB
    # =========================================================

    def create_medical_tab(self):

        self.medical_tab = ttk.Frame(self.notebook)

        self.notebook.add(
            self.medical_tab,
            text="Medical Data"
        )

        # Conditions
        cond_frame = ttk.LabelFrame(
            self.medical_tab,
            text="Conditions",
            padding=10
        )

        cond_frame.pack(
            fill="both",
            expand=True,
            padx=10,
            pady=5
        )

        self.conditions_list = self.styled_listbox(
            cond_frame,
            height=6
        )

        self.conditions_list.pack(
            side="left",
            fill="both",
            expand=True
        )

        tk.Button(
            cond_frame,
            text="Add",
            command=self.add_condition,
            **self.button_style
        ).pack(side="right", padx=5)

        # Medications
        med_frame = ttk.LabelFrame(
            self.medical_tab,
            text="Medications",
            padding=10
        )

        med_frame.pack(
            fill="both",
            expand=True,
            padx=10,
            pady=5
        )

        self.medications_list = self.styled_listbox(
            med_frame,
            height=6
        )

        self.medications_list.pack(
            side="left",
            fill="both",
            expand=True
        )

        tk.Button(
            med_frame,
            text="Add",
            command=self.add_medication,
            **self.button_style
        ).pack(side="right", padx=5)

    # =========================================================
    # NOTES TAB
    # =========================================================

    def create_notes_tab(self):

        self.notes_tab = ttk.Frame(self.notebook)

        self.notebook.add(
            self.notes_tab,
            text="Nursing Notes"
        )

        frame = ttk.LabelFrame(
            self.notes_tab,
            text="Nursing Notes",
            padding=15
        )

        frame.pack(
            fill="both",
            expand=True,
            padx=10,
            pady=10
        )

        self.notes_text = self.styled_text(
            frame,
            height=15
        )

        self.notes_text.pack(
            fill="both",
            expand=True
        )

        tk.Button(
            frame,
            text="Start Recording",
            command=self.start_listening,
            **self.button_style
        ).pack(side="left", padx=5, pady=10)

        tk.Button(
            frame,
            text="Stop Recording",
            command=self.stop_listening,
            **self.button_style
        ).pack(side="left", padx=5, pady=10)

    # =========================================================
    # TRANSLATOR TAB
    # =========================================================

    # =========================================================
# TRANSLATOR TAB
# =========================================================

    def create_translator_tab(self):
    
        self.translator_tab = ttk.Frame(self.notebook)
    
        self.notebook.add(
            self.translator_tab,
            text="Language Translator"
        )
    
        frame = ttk.Frame(
            self.translator_tab,
            padding=15
        )
    
        frame.pack(
            fill="both",
            expand=True
        )

        # =====================================================
        # LANGUAGE SELECTION FRAME
        # =====================================================
    
        lang_frame = ttk.Frame(frame)
    
        lang_frame.pack(
            fill="x",
            pady=10
        )

        # -------------------------
        # FROM LANGUAGE
        # -------------------------
    
        ttk.Label(
            lang_frame,
            text="From Language"
        ).grid(
            row=0,
            column=0,
            padx=10,
            pady=5,
            sticky="w"
        )

        self.from_language = tk.StringVar()
    
        from_combo = ttk.Combobox(
            lang_frame,
            textvariable=self.from_language,
            values=list(languages.keys()),
            state="readonly",
            width=25
        )
    
        from_combo.grid(
            row=1,
            column=0,
            padx=10,
            pady=5
        )

        from_combo.set("English")
    
        # -------------------------
        # TO LANGUAGE
        # -------------------------
    
        ttk.Label(
            lang_frame,
            text="To Language"
        ).grid(
            row=0,
            column=1,
            padx=10,
            pady=5,
            sticky="w"
        )

        self.to_language = tk.StringVar()
    
        to_combo = ttk.Combobox(
            lang_frame,
            textvariable=self.to_language,
            values=list(languages.keys()),
            state="readonly",
            width=25
        )
    
        to_combo.grid(
            row=1,
            column=1,
            padx=10,
            pady=5
        )
    
        to_combo.set("Spanish")

        # =====================================================
        # INPUT TEXT
        # =====================================================
    
        ttk.Label(
            frame,
            text="Input Text"
        ).pack(anchor="w")
    
        self.input_text = self.styled_text(
            frame,
            height=6
        )
    
        self.input_text.pack(
            fill="x",
            pady=5
        )

        # =====================================================
        # TRANSLATE BUTTON
        # =====================================================
    
        tk.Button(
            frame,
            text="Translate",
            command=self.translate_text,
            **self.button_style
        ).pack(pady=10)
    
        # =====================================================
        # OUTPUT TEXT
        # =====================================================
    
        ttk.Label(
            frame,
            text="Translated Text"
        ).pack(anchor="w")
    
        self.output_text = self.styled_text(
            frame,
            height=6
        )
    
        self.output_text.pack(
            fill="x",
            pady=5
        )

    # =========================================================
    # ADD CONDITION
    # =========================================================

    def add_condition(self):

        win = tk.Toplevel(self.root)

        win.title("Add Condition")

        self.style_toplevel(win)

        ttk.Label(
            win,
            text="Name"
        ).pack(pady=5)

        name = self.styled_entry(win)

        name.pack(pady=5)

        ttk.Label(
            win,
            text="Status"
        ).pack(pady=5)

        status = self.styled_entry(win)

        status.pack(pady=5)

        def save():

            self.conditions_list.insert(
                tk.END,
                f"{name.get()} ({status.get()})"
            )

            win.destroy()

        tk.Button(
            win,
            text="Save",
            command=save,
            **self.button_style
        ).pack(pady=10)

    # =========================================================
    # ADD MEDICATION
    # =========================================================

    def add_medication(self):

        win = tk.Toplevel(self.root)

        win.title("Add Medication")

        self.style_toplevel(win)

        ttk.Label(
            win,
            text="Name"
        ).pack(pady=5)

        name = self.styled_entry(win)

        name.pack(pady=5)

        ttk.Label(
            win,
            text="Dosage"
        ).pack(pady=5)

        dosage = self.styled_entry(win)

        dosage.pack(pady=5)

        def save():

            self.medications_list.insert(
                tk.END,
                f"{name.get()} - {dosage.get()}"
            )

            win.destroy()

        tk.Button(
            win,
            text="Save",
            command=save,
            **self.button_style
        ).pack(pady=10)

    # =========================================================
    # TRANSLATE
    # =========================================================

    def translate_text(self):
    
        try:
    
            text = self.input_text.get(
                "1.0",
                tk.END
            ).strip()
    
            if not text:
    
                messagebox.showwarning(
                    "Warning",
                    "Please enter text to translate."
                )
    
                return
    
            # Get selected languages
            from_lang_name = self.from_language.get()
            to_lang_name = self.to_language.get()
    
            # Convert names to language codes
            source_lang = languages[from_lang_name]
            target_lang = languages[to_lang_name]
    
            # Translate
            translated = GoogleTranslator(
                source=source_lang,
                target=target_lang
            ).translate(text)
    
            self.output_text.delete(
                "1.0",
                tk.END
            )
    
            self.output_text.insert(
                tk.END,
                translated
            )
    
        except Exception as e:
    
            messagebox.showerror(
                "Translation Error",
                str(e)
            )

    # =========================================================
    # VALIDATE
    # =========================================================

    def validate(self):

        if not self.fields["patient_id"].get().strip():

            messagebox.showerror(
                "Validation Error",
                "Patient ID required"
            )

            return False

        return True

    # =========================================================
    # SAVE JSON
    # =========================================================

    def save_json(self):

        if not self.validate():
            return

        data = {
            "patient_id": self.fields["patient_id"].get(),
            "first_name": self.fields["first_name"].get(),
            "last_name": self.fields["last_name"].get(),
            "email": self.fields["email"].get(),
            "phone": self.fields["phone"].get(),
            "gender": self.gender_var.get(),
            "notes": self.notes_text.get("1.0", tk.END).strip()
        }

        path = filedialog.asksaveasfilename(
            defaultextension=".json",
            filetypes=[("JSON Files", "*.json")]
        )

        if not path:
            return

        with open(path, "w") as f:

            json.dump(
                data,
                f,
                indent=4
            )

        messagebox.showinfo(
            "Saved",
            "Patient record saved successfully!"
        )

    # =========================================================
    # LOAD JSON
    # =========================================================

    def load_json(self):

        path = filedialog.askopenfilename(
            filetypes=[("JSON Files", "*.json")]
        )

        if not path:
            return

        with open(path) as f:

            data = json.load(f)

        self.fields["patient_id"].delete(0, tk.END)
        self.fields["patient_id"].insert(0, data.get("patient_id", ""))

        self.fields["first_name"].delete(0, tk.END)
        self.fields["first_name"].insert(0, data.get("first_name", ""))

        self.fields["last_name"].delete(0, tk.END)
        self.fields["last_name"].insert(0, data.get("last_name", ""))

        self.fields["email"].delete(0, tk.END)
        self.fields["email"].insert(0, data.get("email", ""))

        self.fields["phone"].delete(0, tk.END)
        self.fields["phone"].insert(0, data.get("phone", ""))

        self.gender_var.set(
            data.get("gender", "Male")
        )

        self.notes_text.delete(
            "1.0",
            tk.END
        )

        self.notes_text.insert(
            tk.END,
            data.get("notes", "")
        )

        messagebox.showinfo(
            "Loaded",
            "Patient record loaded successfully!"
        )

    # =========================================================
    # GENERATE PDF
    # =========================================================

    def generate_pdf(self):

        try:

            path = filedialog.asksaveasfilename(
                defaultextension=".pdf",
                filetypes=[("PDF Files", "*.pdf")]
            )

            if not path:
                return

            doc = SimpleDocTemplate(
                path,
                pagesize=letter
            )

            styles = getSampleStyleSheet()

            elements = []

            title = Paragraph(
                "<b>Electronic Medical Record</b>",
                styles["Title"]
            )

            elements.append(title)

            elements.append(
                Spacer(1, 20)
            )

            patient_info = f"""
            <b>Patient ID:</b> {self.fields['patient_id'].get()}<br/>
            <b>First Name:</b> {self.fields['first_name'].get()}<br/>
            <b>Last Name:</b> {self.fields['last_name'].get()}<br/>
            <b>Email:</b> {self.fields['email'].get()}<br/>
            <b>Phone:</b> {self.fields['phone'].get()}<br/>
            """

            elements.append(
                Paragraph(
                    patient_info,
                    styles["BodyText"]
                )
            )

            notes = self.notes_text.get(
                "1.0",
                tk.END
            ).strip()

            elements.append(
                Spacer(1, 20)
            )

            elements.append(
                Paragraph(
                    "<b>Nursing Notes</b>",
                    styles["Heading2"]
                )
            )

            elements.append(
                Paragraph(
                    notes,
                    styles["BodyText"]
                )
            )

            doc.build(elements)

            messagebox.showinfo(
                "Success",
                "PDF generated successfully!"
            )

        except Exception as e:

            messagebox.showerror(
                "PDF Error",
                str(e)
            )

    # =========================================================
    # SPEECH To TEXT
    # =========================================================

    def speech_to_text(self):
     
        # Open recorded audio
        wf = wave.open("recorded.wav", "rb")
    
        # Initialize recognizer
        rec = KaldiRecognizer(model, wf.getframerate())
    
        transcription = ""
    
        # Transcribe
        while True:
            data = wf.readframes(4000)
    
            if len(data) == 0:
                break
    
            if rec.AcceptWaveform(data):
                result = json.loads(rec.Result())
                transcription += result.get("text", "") + " "
    
        # Final result
        result = json.loads(rec.FinalResult())
        transcription += result.get("text", "")

        corrected = TextBlob(transcription)
        
        #text_notes.insert(tk.END, transcription)
        self.notes_text.insert(tk.END, corrected)
    
        return corrected
    
    
    # =========================================================
    # SPEECH THREAD
    # =========================================================

    def audio_thread(self):

        global listening, audio_frames

        rec = KaldiRecognizer(
            model,
            16000
        )

        audio_frames = []

        def callback(indata, frames, time, status):

            if listening:

                data = bytes(indata)

                audio_frames.append(data)

                if rec.AcceptWaveform(data):

                    result = json.loads(
                        rec.Result()
                    )

                    q.put(
                        result.get("text", "")
                    )

        with sd.RawInputStream(
            samplerate=16000,
            blocksize=8000,
            dtype='int16',
            channels=1,
            callback=callback
        ):

            while listening:
                sd.sleep(100)

    # =========================================================
    # UPDATE TEXTBOX
    # =========================================================

    def update_textbox(self):

        try:

            while True:

                text = q.get_nowait()

                self.notes_text.insert(
                    tk.END,
                    text + "\n"
                )

                self.notes_text.see(tk.END)

        except queue.Empty:
            pass

        self.root.after(
            100,
            self.update_textbox
        )

    # =========================================================
    # SAVE WAV
    # =========================================================

    def save_wav(self, filename="recorded.wav"):

        wf = wave.open(
            filename,
            'wb'
        )

        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(16000)

        wf.writeframes(
            b''.join(audio_frames)
        )

        wf.close()

    # =========================================================
    # START LISTENING
    # =========================================================

    def start_listening(self):

        global listening

        listening = True

        print("Listening...")

        threading.Thread(
            target=self.audio_thread,
            daemon=True
        ).start()

    # =========================================================
    # STOP LISTENING
    # =========================================================

    def stop_listening(self):

        global listening

        listening = False

        self.save_wav()

        print("Saved to recorded.wav")

        messagebox.showinfo(
            "Recording",
            "Speech recording stopped."
        )

        self.speech_to_text()

        
# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    root = tk.Tk()

    app = MedicalApp(root)

    root.mainloop()

Listening...
Saved to recorded.wav


In [16]:
import sys

!{sys.executable} -m pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------- ----------------------- 0.8/2.0 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 6.0 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: D:\LLM\cuda\Scripts\python.exe -m pip install --upgrade pip


In [17]:
from reportlab.lib.pagesizes import letter

print("ReportLab works")

ReportLab works
